In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import sys
import hydra
from omegaconf import OmegaConf
import wandb

# Add the project source directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import project modules
from src.data.dataset import load_mnist
from src.train import get_target_distributions
from src.utils.losses import total_loss_fn

# Suppress Hydra's output in the notebook
from hydra.core.global_hydra import GlobalHydra
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. Configuration

Load the configuration using OmegaConf. We'll merge the main `config.yaml` with our new `quick_test.yaml` to set up a short training run.

In [ ]:
conf_path = os.path.join(project_root, 'conf')

# Start with the base config
cfg = OmegaConf.load(os.path.join(conf_path, 'config.yaml'))

# Define which specific configs to load
# Using dglow_resnet as an example, can be changed
model_config_name = "dglow_resnet.yaml" 
data_config_name = "mnist.yaml"
# Use the new quick_test config for a short run
training_config_name = "quick_test.yaml"

# Create a structured config by merging the components
cfg = OmegaConf.merge(
    cfg,
    OmegaConf.load(os.path.join(conf_path, 'model', model_config_name)),
    OmegaConf.load(os.path.join(conf_path, 'data', data_config_name)),
    OmegaConf.load(os.path.join(conf_path, 'training', training_config_name)),
)

# Disable wandb for this quick test to avoid clutter
cfg.wandb.mode = "disabled"

print("Configuration loaded for a quick test:")
print(f"Epochs: {cfg.training.epochs}")
print(f"Model: {cfg.model._target_}")
print(f"lr_vars: {cfg.training.lr_vars}")

### 2. Data Loading

In [ ]:
# The load_mnist function from the project returns DataLoaders.
cfg.data.dataset.path = os.path.join(project_root, cfg.data.dataset.path)
train_loader, test_loader = load_mnist(cfg.data)

print(f"Loaded MNIST train dataset with {len(train_loader.dataset)} samples.")
print(f"Loaded MNIST test dataset with {len(test_loader.dataset)} samples.")

### 3. Model and Optimizer Setup

Here, we initialize the model, `trainable_means`, and `trainable_v`. We explicitly set `requires_grad` to `False` for parameters with a learning rate of 0.

In [ ]:
# Initialize model
model = hydra.utils.instantiate(cfg.model, _convert_="partial").to(device)

# Initialize trainable means and variances
with torch.no_grad():
    initial_means = torch.zeros(cfg.training.num_classes, cfg.training.features, device=device)
    for i in range(cfg.training.num_classes):
        initial_means[i, i] = cfg.training.latent_separation
    initial_means += torch.randn_like(initial_means) * cfg.training.latent_noise
    trainable_means = nn.Parameter(initial_means)

    initial_v = torch.ones(cfg.training.num_classes, cfg.training.features, device=device) * torch.log(torch.tensor(cfg.training.latent_v))
    initial_v += torch.randn_like(initial_v) * cfg.training.latent_noise
    trainable_v = nn.Parameter(initial_v)

# Freeze parameters if their learning rate is 0
if cfg.training.lr_means == 0:
    trainable_means.requires_grad_(False)
if cfg.training.lr_vars == 0:
    trainable_v.requires_grad_(False)

print(f"trainable_means.requires_grad: {trainable_means.requires_grad}")
print(f"trainable_v.requires_grad: {trainable_v.requires_grad}")

# Setup optimizer
# Only add parameters to the optimizer if they require gradients
param_groups = [{'params': model.parameters(), 'lr': cfg.training.lr, 'weight_decay': cfg.training.weight_decay}]
if trainable_means.requires_grad:
    param_groups.append({'params': [trainable_means], 'lr': cfg.training.lr_means, 'weight_decay': 0.0})
if trainable_v.requires_grad:
    param_groups.append({'params': [trainable_v], 'lr': cfg.training.lr_vars, 'weight_decay': 0.0})

optimizer = optim.AdamW(param_groups)

print("\nOptimizer Parameter Groups:")
for i, group in enumerate(optimizer.param_groups):
    print(f"  Group {i}:")
    print(f"    lr: {group['lr']}")
    print(f"    weight_decay: {group['weight_decay']}")
    print(f"    Number of params: {len(group['params'])}")

### 4. Training Loop Test

We'll run the training for just a few batches and check if `trainable_v` changes.

In [ ]:
# Auxiliary layers setup from train.py
aux_layers = np.arange(start=cfg.training.aux_freq - 1, stop=cfg.training.aux_total, step=cfg.training.aux_freq)
alphas = torch.tensor(np.geomspace(start=cfg.training.gamma_alpha ** len(aux_layers), stop=1, num=len(aux_layers)), device=device)
betas = torch.tensor(np.geomspace(start=cfg.training.gamma_beta ** len(aux_layers), stop=1, num=len(aux_layers)), device=device)

model.train()
max_batches = 5  # Limit to 5 batches for a quick check

# Store the initial value of trainable_v to compare against
v_before = trainable_v.clone().detach()

for i, (x_batch, y_batch) in enumerate(train_loader):
    if i >= max_batches:
        break
        
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    optimizer.zero_grad()

    # Get target distributions
    target_dists = get_target_distributions(trainable_means, trainable_v, cfg.training.num_classes)

    # Forward pass
    intermediate_outputs = model(x_batch)
    
    # Loss calculation
    loss, _, _ = total_loss_fn(intermediate_outputs, y_batch, target_dists, cfg, alphas, betas, aux_layers)
    
    loss.backward()
    optimizer.step()
    
    print(f"Batch {i+1}/{max_batches} - Loss: {loss.item():.4f}")

# Check if trainable_v has changed
v_after = trainable_v.clone().detach()
v_diff = torch.abs(v_before - v_after).sum()

print("\n--- Verification ---")
print(f"Sum of absolute difference in trainable_v before and after training: {v_diff.item()}")

if v_diff == 0:
    print("✅ SUCCESS: trainable_v was not updated, as expected.")
else:
    print("❌ FAILURE: trainable_v was updated unexpectedly.")